# GerDraCor Sound Annotation — Postprocessing

This notebook cleans the raw predictions produced by the sound annotation model **before** they are evaluated against the gold standard or used for further analysis.

It applies three postprocessing steps, in order, directly to the XML files:

1. **Remove known false-positive annotations** — deletes `<character_sound>`/`<ambient_sound>` tags around phrases that are known, recurring mistakes of the model.
2. **Merge split annotations** — joins two annotations of the same type that were incorrectly split across a line break.
3. **Remove sound tags around speaker names** — the model sometimes wraps a speaker's name (e.g. `WENDLA.`) in a sound tag by mistake; this step removes the tag while keeping the name as plain text.

**⚠️ Files are overwritten in place.** Make a copy of your prediction folder before running this notebook if you want to keep the unprocessed version.

Run this notebook **before** `02_evaluation.ipynb`. The evaluation notebook expects to read already-postprocessed files.

## 0. Setup

In [193]:
import re
from pathlib import Path

print('Imports OK')

Imports OK


## 1. Configuration

Set the folder containing the model's raw predictions, and the path to the false-positive trigger list.

In [194]:
# ── CONFIGURE PATHS HERE ──────────────────────────────────────────────────────

# Folder containing the predicted XML files (will be overwritten in place)
#PRED_DIR = '/Users/sguhr/Desktop/Sound_in_Drama/20260621_data_postprocessed'
PRED_DIR = '/Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_all_prose_model_predicted_plays_augmented/exp_all_prose_predicted_augmented_postprocessed'

# Text file with one false-positive trigger phrase per line (see Step 1 below)
TRIGGERS_FILE = '/Users/sguhr/Desktop/Sound_in_Drama/frequent_false_positives.txt'

# Sound tag names present in the files
SOUND_TAGS = ('character_sound', 'ambient_sound')

PREDICTIONS_FOLDER = Path(PRED_DIR)
OUTPUT_FOLDER = Path(PRED_DIR)  # overwrite in place

def count_sound_tags(folder: Path) -> int:
    """Count total <character_sound>/<ambient_sound> opening tags across all XML files in *folder*."""
    total = 0
    pattern = re.compile(r'<(?:character_sound|ambient_sound)(?:\s[^>]*)?>')
    for xml_file in folder.glob('*.xml'):#for xml_file in folder.rglob('*.xml'): #add "r" to glob when recursing subforlders
        with open(xml_file, encoding='utf-8') as f:
            total += len(pattern.findall(f.read()))
    return total

n_files = len(list(PREDICTIONS_FOLDER.glob('*.xml')))
n_before = count_sound_tags(PREDICTIONS_FOLDER)
print(f'Found {n_files} XML files in {PRED_DIR}')
print(f'Total sound annotations before postprocessing: {n_before}')

Found 12 XML files in /Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_all_prose_model_predicted_plays_augmented/exp_all_prose_predicted_augmented_postprocessed
Total sound annotations before postprocessing: 1836


## 2. Step 0 — Remove Augmentation Artifacts ("sagt"/"sagen") from Stage Direction Openings

Before annotation, a preprocessing heuristic (see the accompanying paper, Section 3.1) inserted the word "sagt" or "sagen" at the start of elliptical stage directions that lacked a finite verb -- for example, turning `sehr ruhig.` into `sagt sehr ruhig.` -- so the sound-annotation model would have a syntactically complete verbal phrase to work with.

This inserted word is **not part of the original play text**. Before any further postprocessing, this step removes it again from the start of a `<stage>` element (optionally followed immediately by a `<character_sound>` or `<ambient_sound>` opening tag), restoring the original elliptical stage direction:

```xml
<stage>sagt sehr ruhig.</stage>                              ->  <stage>sehr ruhig.</stage>
<stage><character_sound>sagt laut</character_sound></stage>  ->  <stage><character_sound>laut</character_sound></stage>
```

**Matching is case-sensitive** (lowercase "sagt"/"sagen" only): the augmentation heuristic always inserted these words in lowercase, so a capitalized occurrence (e.g. "Sagt" at the start of a genuinely complete original sentence) is assumed to be original text and is left untouched. A word boundary check also prevents partial matches such as "sagend" or "Ansage".

In [195]:
# Matches one of three openings, followed by optional whitespace and the bare
# word "sagt" or "sagen" (word-boundary protected, case-sensitive).
#
'''
# Group 1 -- the opening tag sequence to keep
# Group 2 -- "sagt" or "sagen" to delete
#SAGT_SAGEN_PATTERN = re.compile(
    r"""
    (                                       # Group 1: opening to keep
        <stage>                             #   bare <stage>
        (?:                                 #   optionally followed by
            \s*                             #     optional whitespace
            <(?:character_sound|ambient_sound)>  # a sound open tag
        )?
    )
    \s*                                     # optional whitespace before word
    (sagt|sagen)                            # Group 2: word to DELETE (lowercase only)
    \b                                      # word boundary (not "sagend" etc.)
    \s*                                     # optional whitespace after word
    """,
    re.VERBOSE,  # NOTE: re.IGNORECASE intentionally omitted -- case-sensitive match
)


def remove_sagt_sagen_files(input_folder: Path, output_folder: Path) -> int:
    """Remove augmentation-inserted sagt/sagen from stage-direction openings.

    Returns the total number of occurrences removed across all files.
    """
    output_folder.mkdir(parents=True, exist_ok=True)
    xml_files = list(input_folder.glob('*.xml'))
    print(f'Removing augmentation artifacts in {len(xml_files)} XML files ...')

    total_removed = 0
    for xml_file in xml_files:
        with open(xml_file, encoding='utf-8') as f:
            content = f.read()

        cleaned, n = SAGT_SAGEN_PATTERN.subn(r'\1', content)
        total_removed += n

        out_path = output_folder / xml_file.name
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        if n:
            print(f'  {xml_file.name}: removed {n} occurrence(s)')

    print(f'\nStep 0 done. {total_removed} occurrence(s) removed.')
    return total_removed

'''
#step0_removed = remove_sagt_sagen_files(PREDICTIONS_FOLDER, OUTPUT_FOLDER)

'\n# Group 1 -- the opening tag sequence to keep\n# Group 2 -- "sagt" or "sagen" to delete\n#SAGT_SAGEN_PATTERN = re.compile(\n    r"""\n    (                                       # Group 1: opening to keep\n        <stage>                             #   bare <stage>\n        (?:                                 #   optionally followed by\n            \\s*                             #     optional whitespace\n            <(?:character_sound|ambient_sound)>  # a sound open tag\n        )?\n    )\n    \\s*                                     # optional whitespace before word\n    (sagt|sagen)                            # Group 2: word to DELETE (lowercase only)\n    \x08                                      # word boundary (not "sagend" etc.)\n    \\s*                                     # optional whitespace after word\n    """,\n    re.VERBOSE,  # NOTE: re.IGNORECASE intentionally omitted -- case-sensitive match\n)\n\n\ndef remove_sagt_sagen_files(input_folder: Path, output_folder: P

## 2. Step 1 — Remove Known False-Positive Annotations

The model makes some **recurring, predictable mistakes** — for example, annotating a bare name ("Vater", "Frau") or a single punctuation mark as a sound event. These are collected in `frequent_false_positives.txt`, one phrase per line.

For each phrase in the list, this step removes the `<character_sound>` or `<ambient_sound>` tags wherever their text content is an **exact match** (case-insensitive) for that phrase. Only the tags are removed — the underlying text is always kept, so no words are deleted from the play.

**To extend the list:** open `frequent_false_positives.txt` and add a new phrase on its own line. No code changes are needed.

In [196]:
def load_false_positive_triggers(path: str) -> list[str]:
    """Load one trigger phrase per line from a text file.

    A blank line represents the empty string, which matches fully empty
    annotations such as <character_sound></character_sound>. Only a single
    trailing blank line (the artifact of the file ending in a newline) is
    dropped; intentional blank lines elsewhere in the file are kept.
    """
    with open(path, encoding='utf-8') as f:
        lines = f.read().split('\n')
    if lines and lines[-1] == '':
        lines = lines[:-1]
    return lines


FALSE_POSITIVE_TRIGGERS = load_false_positive_triggers(TRIGGERS_FILE)
print(f'Loaded {len(FALSE_POSITIVE_TRIGGERS)} false-positive triggers from {TRIGGERS_FILE}')

Loaded 638 false-positive triggers from /Users/sguhr/Desktop/Sound_in_Drama/frequent_false_positives.txt


In [197]:
def remove_annotations_for_triggers(xml_content: str, triggers: list[str]) -> str:
    """Remove <character_sound> or <ambient_sound> tags around trigger phrases."""
    cleaned = xml_content
    for trigger in triggers:
        escaped = re.escape(trigger)
        pattern = (
            r'<(character_sound|ambient_sound)>'
            r'(\s*)'
            r'(' + escaped + r')'
            r'(\s*)'
            r'</\1>'
        )
        cleaned = re.sub(pattern, r'\2\3\4', cleaned, flags=re.IGNORECASE | re.DOTALL)
    return cleaned


def clean_prediction_files(input_folder: Path, output_folder: Path, triggers: list[str]) -> int:
    """Process all XML files in the input folder and save cleaned versions.

    Returns the total number of annotations removed.
    """
    output_folder.mkdir(parents=True, exist_ok=True)
    xml_files = list(input_folder.glob('*.xml'))
    print(f'Found {len(xml_files)} XML files in {input_folder}')

    total_removed = 0
    for xml_file in xml_files:
        with open(xml_file, encoding='utf-8') as f:
            content = f.read()

        cleaned = remove_annotations_for_triggers(content, triggers)

        orig_count = len(re.findall(r'<(character_sound|ambient_sound)>', content))
        new_count = len(re.findall(r'<(character_sound|ambient_sound)>', cleaned))
        removed = orig_count - new_count
        total_removed += removed

        output_path = output_folder / xml_file.name
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        if removed > 0:
            print(f'  {xml_file.name}: removed {removed} annotation(s)')

    print(f'\nStep 1 done. {total_removed} false-positive annotation(s) removed.')
    return total_removed


step1_removed = clean_prediction_files(PREDICTIONS_FOLDER, OUTPUT_FOLDER, FALSE_POSITIVE_TRIGGERS)

Found 12 XML files in /Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_all_prose_model_predicted_plays_augmented/exp_all_prose_predicted_augmented_postprocessed
  chezy-der-neue-narziss.xml: removed 7 annotation(s)
  guenderode-udohla.xml: removed 4 annotation(s)
  schiller-die-raeuber.xml: removed 41 annotation(s)
  lessing-emilia-galotti.xml: removed 16 annotation(s)
  dohm-ein-schuss-ins-schwarze.xml: removed 7 annotation(s)
  wedekind-fruehlings-erwachen.xml: removed 28 annotation(s)
  ebner-eschenbach-die-veilchen.xml: removed 9 annotation(s)
  borchert-draussen-vor-der-tuer.xml: removed 46 annotation(s)
  dovsky-mona-lisa.xml: removed 1 annotation(s)
  sachs-eulenspiegel-mit-dem-blauen-hosentuch.xml: removed 1 annotation(s)
  neuber-die-beschuetzte-schauspielkunst.xml: removed 3 annotation(s)
  ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml: removed 3 annotation(s)

Step 1 done. 166 false-positive

In [198]:
# ── Diagnostic: why did Step 1 remove 0 annotations? ─────────────────────────
import re
from collections import Counter

SOUND_ELEM_TEXT = re.compile(
    r'<(?:character_sound|ambient_sound)(?:\s[^>]*)?>([^<]*)</(?:character_sound|ambient_sound)>',
    re.IGNORECASE,
)

trigger_set_lower = {t.strip().lower() for t in FALSE_POSITIVE_TRIGGERS if t.strip()}

all_span_texts = Counter()
for xml_file in PREDICTIONS_FOLDER.glob('*.xml'):
    with open(xml_file, encoding='utf-8') as f:
        content = f.read()
    for span_text in SOUND_ELEM_TEXT.findall(content):
        normalised = ' '.join(span_text.strip().split())  # collapse all whitespace to single spaces
        all_span_texts[normalised.lower()] += 1

print(f'Total distinct annotated spans found: {len(all_span_texts)}')
print(f'Total trigger phrases loaded: {len(trigger_set_lower)}')

exact_overlap = set(all_span_texts) & trigger_set_lower
print(f'\nSpans that exactly match a trigger (whitespace-normalised): {len(exact_overlap)}')
for span in sorted(exact_overlap)[:10]:
    print(f'  "{span}"  (appears {all_span_texts[span]}x)')

print('\nMost frequent annotated spans in this dataset (check manually for false positives):')
for span, count in all_span_texts.most_common(20):
    in_list = '✓ in trigger list' if span in trigger_set_lower else ''
    print(f'  {count:3d}x  "{span}"  {in_list}')

Total distinct annotated spans found: 1175
Total trigger phrases loaded: 619

Spans that exactly match a trigger (whitespace-normalised): 0

Most frequent annotated spans in this dataset (check manually for false positives):
   77x  "sagt"  
   19x  "der prinz"  
   14x  "sag ich"  
   13x  "eva"  
   12x  "sagt ich"  
   12x  "elßbeth"  
   11x  "laut"  
   11x  "amalia"  
   11x  "marinelli"  
   11x  "marie"  
   10x  "franz"  
   10x  "odoardo"  
    9x  "ich sage dir"  
    8x  "orsina"  
    8x  "laura"  
    8x  "spricht"  
    8x  "fridbert"  
    7x  "moor"  
    7x  "ich bitte dich"  
    7x  "emilia"  


## 3. Step 2 — Merge Consecutive Same-Type Annotations Split by Whitespace or a Page Break

Sometimes the model splits a single continuous sound event into **two separate annotations** of the same type, purely because something structural interrupts the text — a line break, a plain space, or a `<pb/>` page-break element. For example:

```xml
<ambient_sound>Glocken</ambient_sound> <ambient_sound>läuten</ambient_sound>

<ambient_sound>Seufzen und</ambient_sound> <pb n="1592"/> <ambient_sound>Schreien</ambient_sound>
```

This step detects pairs of same-type sound elements (`character_sound`+`character_sound` or `ambient_sound`+`ambient_sound`) separated only by whitespace and/or a `<pb/>` element, and merges them into a single span:

```xml
<ambient_sound>Glocken läuten</ambient_sound>

<ambient_sound>Seufzen und Schreien</ambient_sound>
```

`<pb/>` is bridged over.

Whitespace (including line breaks) is otherwise preserved exactly inside the merged span, so character offsets stay as close as possible to the original. Only duplicate horizontal whitespace is collapsed to a single space. The merge runs in a loop so chains of three or more consecutive split annotations are fully collapsed.

**This step requires no vocabulary or word list** — it merges purely on structural grounds (same tag type + a non-content separator), so it generalises safely to any text without needing to anticipate spelling variants or vocabulary.

In [199]:
# Separator: one or more runs of whitespace (space/tab/newline) and/or <pb/> elements,
# in any combination. No vocabulary involved -- purely structural.
SEPARATOR = r'(?:[ \t\n]+|<pb(?:\s[^>]*)?/?>)+'

MERGE_PATTERN = re.compile(
    r'<(character_sound|ambient_sound)(\s[^>]*)?>([^<]*)</(?:character_sound|ambient_sound)>'
    r'(' + SEPARATOR + r')'
    r'<(character_sound|ambient_sound)(\s[^>]*)?>([^<]*)</(?:character_sound|ambient_sound)>',
    re.IGNORECASE,
)


def merge_consecutive_annotations(content: str) -> tuple[str, int]:
    """Repeatedly merge consecutive same-type sound elements split by whitespace and/or a <pb/> tag.

    The separator (including any <pb/> element) is preserved exactly as-is inside the
    merged span, so page-break markers and whitespace are not lost.

    Returns (cleaned_content, total_number_of_merges_performed).
    """
    total_merges = 0
    while True:
        merges_this_pass = 0

        def _merge(m: re.Match) -> str:
            nonlocal merges_this_pass
            tag1, attrs1, text1 = m.group(1), m.group(2) or '', m.group(3)
            separator = m.group(4)
            tag2, text2 = m.group(5), m.group(7)
            if tag1.lower() != tag2.lower():
                return m.group(0)
            merges_this_pass += 1
            return f'<{tag1}{attrs1}>{text1}{separator}{text2}</{tag1}>'

        content = MERGE_PATTERN.sub(_merge, content)
        total_merges += merges_this_pass
        if merges_this_pass == 0:
            break
    return content, total_merges


def merge_prediction_files(input_folder: Path, output_folder: Path) -> int:
    """Apply consecutive-annotation merging to all XML files. Returns total merges."""
    output_folder.mkdir(parents=True, exist_ok=True)
    xml_files = list(input_folder.glob('*.xml'))
    print(f'Merging consecutive annotations in {len(xml_files)} XML files ...')

    total_merges = 0
    for xml_file in xml_files:
        with open(xml_file, encoding='utf-8') as f:
            content = f.read()

        cleaned, merges = merge_consecutive_annotations(content)
        total_merges += merges

        out_path = output_folder / xml_file.name
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        if merges:
            print(f'  {xml_file.name}: merged {merges} annotation pair(s)')

    print(f'\nStep 2 done. {total_merges} annotation pair(s) merged.')
    return total_merges


step2_merged = merge_prediction_files(PREDICTIONS_FOLDER, OUTPUT_FOLDER)

Merging consecutive annotations in 12 XML files ...
  schiller-die-raeuber.xml: merged 41 annotation pair(s)
  lessing-emilia-galotti.xml: merged 1 annotation pair(s)
  dohm-ein-schuss-ins-schwarze.xml: merged 6 annotation pair(s)
  wedekind-fruehlings-erwachen.xml: merged 6 annotation pair(s)
  ebner-eschenbach-die-veilchen.xml: merged 1 annotation pair(s)
  borchert-draussen-vor-der-tuer.xml: merged 22 annotation pair(s)
  sachs-eulenspiegel-mit-dem-blauen-hosentuch.xml: merged 4 annotation pair(s)
  neuber-die-beschuetzte-schauspielkunst.xml: merged 1 annotation pair(s)

Step 2 done. 82 annotation pair(s) merged.


## 4. Step 3 — Remove Sound Tags Around Speaker Names

### What this fixes

In TEI-encoded plays, every line of dialogue is preceded by a `<speaker>` element giving the character's name, e.g.:

```xml
<speaker>WENDLA.</speaker>
```

The auto-annotation model occasionally wraps this speaker name itself in a `<character_sound>` tag:

```xml
<speaker><character_sound>WENDLA.</character_sound></speaker>
```

A speaker name is **not** a sound event — it is metadata identifying who speaks next. This is a systematic, structural error rather than a content-based one, so it cannot be caught by the phrase list in Step 1 (which would need a separate entry for every character name in the entire corpus).

### What this step does

This step scans every `<speaker>...</speaker>` block in each file and removes any `<character_sound>` or `<ambient_sound>` tags found directly inside it, while keeping the speaker's name as plain text. The above example becomes:

```xml
<speaker>WENDLA.</speaker>
```

Set `EXCLUDE_SPEAKER_ANNOTATIONS = False` below to skip this step (for example, if your prediction files do not show this error pattern).

In [200]:
# Set to True to remove sound-tag wrapping around speaker names (see explanation above)
EXCLUDE_SPEAKER_ANNOTATIONS = True

SPEAKER_BLOCK = re.compile(r'(<speaker>)(.*?)(</speaker>)', re.IGNORECASE | re.DOTALL)
SOUND_WRAP    = re.compile(r'<(?:character_sound|ambient_sound)>(.*?)</(?:character_sound|ambient_sound)>',
                            re.IGNORECASE | re.DOTALL)


def strip_speaker_sound_wrapping(content: str) -> tuple[str, int]:
    """Remove sound tags wrapped directly around <speaker> content.

    Only tags found *inside* a <speaker>...</speaker> block are affected;
    sound annotations elsewhere in the file (stage directions, dialogue) are untouched.
    Returns (cleaned_content, number_of_tags_removed).
    """
    total_removed = 0

    def _clean_block(m: re.Match) -> str:
        nonlocal total_removed
        open_tag, inner, close_tag = m.group(1), m.group(2), m.group(3)
        new_inner, n = SOUND_WRAP.subn(r'\1', inner)
        total_removed += n
        return open_tag + new_inner + close_tag

    cleaned = SPEAKER_BLOCK.sub(_clean_block, content)
    return cleaned, total_removed


def exclude_speaker_annotation_files(input_folder: Path, output_folder: Path) -> int:
    """Apply speaker-name unwrapping to all XML files. Returns total tags removed."""
    output_folder.mkdir(parents=True, exist_ok=True)
    xml_files = list(input_folder.glob('*.xml'))
    print(f'Removing sound tags around speaker names in {len(xml_files)} XML files ...')

    total_removed = 0
    for xml_file in xml_files:
        with open(xml_file, encoding='utf-8') as f:
            content = f.read()

        cleaned, removed = strip_speaker_sound_wrapping(content)
        total_removed += removed

        out_path = output_folder / xml_file.name
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        if removed:
            print(f'  {xml_file.name}: removed {removed} speaker-wrapped tag(s)')

    print(f'\nStep 3 done. {total_removed} speaker-wrapped sound tag(s) removed.')
    return total_removed


if EXCLUDE_SPEAKER_ANNOTATIONS:
    step3_removed = exclude_speaker_annotation_files(PREDICTIONS_FOLDER, OUTPUT_FOLDER)
else:
    step3_removed = 0
    print('Step 3 skipped (EXCLUDE_SPEAKER_ANNOTATIONS = False).')

Removing sound tags around speaker names in 12 XML files ...
  chezy-der-neue-narziss.xml: removed 1 speaker-wrapped tag(s)
  guenderode-udohla.xml: removed 3 speaker-wrapped tag(s)
  schiller-die-raeuber.xml: removed 53 speaker-wrapped tag(s)
  lessing-emilia-galotti.xml: removed 73 speaker-wrapped tag(s)
  dohm-ein-schuss-ins-schwarze.xml: removed 28 speaker-wrapped tag(s)
  wedekind-fruehlings-erwachen.xml: removed 19 speaker-wrapped tag(s)
  ebner-eschenbach-die-veilchen.xml: removed 16 speaker-wrapped tag(s)
  borchert-draussen-vor-der-tuer.xml: removed 18 speaker-wrapped tag(s)
  sachs-eulenspiegel-mit-dem-blauen-hosentuch.xml: removed 1 speaker-wrapped tag(s)
  ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml: removed 35 speaker-wrapped tag(s)

Step 3 done. 247 speaker-wrapped sound tag(s) removed.


## 6. Step 4 — Reclassify and Merge Fragmented Speech-Verb Runs

### What this fixes

The model sometimes splits a single speech act into several adjacent sound elements of mixed type, separated only by spaces (not a line break), for example:

```xml
<ambient_sound>sagt</ambient_sound> <character_sound>unten mit</character_sound> <ambient_sound>lauter Stimme ausschreiend</ambient_sound>
```

This is wrong in two ways at once: the phrase is artificially cut into three fragments instead of one continuous event, and the bare speech verb ("sagt") is mislabelled as `ambient_sound` -- speech is always character-produced, never ambient.

### What this step does

Inside every `<stage>` element, this step looks for **runs of two or more adjacent sound elements separated only by spaces or tabs** (no line break -- that case is already handled by Step 2). If **any** element in such a run is a single bare verb -- detected via spaCy part-of-speech tagging (`VERB` or `AUX`), not a fixed word list -- the entire run is merged into a single `<character_sound>` span, preserving the original text and spacing exactly:

```xml
<character_sound>sagt unten mit lauter Stimme ausschreiend</character_sound>
```

Runs that contain **no** detected verb are left untouched -- this is the safety net that prevents merging genuinely separate, adjacent sound events (e.g. two distinct ambient sounds mentioned next to each other).

**Why POS tagging instead of a word list:** an earlier version of this step relied on a fixed vocabulary of speech verbs, which would not have generalised to spelling variants, dialect forms, or verbs simply not yet added to the list. Using the same German spaCy model already used elsewhere in this project's stage-direction preprocessing (for finite-verb detection), this step instead checks the grammatical category of the bare word directly -- so it requires no manual list maintenance and generalises automatically to any verb form the model recognises.

**Limitation:** this step only fixes the specific cross-type *mislabelling* problem (a verb tagged as `ambient_sound` when it should be `character_sound`), which is a different issue from the purely structural splitting that Step 2 handles. POS tagging is not perfect -- ambiguous or unusual word forms may occasionally be mis-tagged by the spaCy model, so spot-checking a sample of merged runs is still advisable.

In [201]:
import spacy

# Load the same German spaCy model used elsewhere in this project for finite-verb
# detection (e.g. in the stage-direction augmentation heuristic). Using POS tagging
# instead of a fixed word list means this generalises to any spelling, dialect form,
# or vocabulary without manual maintenance.
print("Loading spaCy German model for finite-verb detection ...")
nlp = spacy.load("de_core_news_sm", disable=["ner", "parser", "lemmatizer"])
print("Model loaded.")

STAGE_BLOCK = re.compile(r'(<stage(?:\s[^>]*)?>)(.*?)(</stage>)', re.IGNORECASE | re.DOTALL)
SOUND_ELEM  = re.compile(
    r'<(character_sound|ambient_sound)(\s[^>]*)?>([^<]*)</(?:character_sound|ambient_sound)>',
    re.IGNORECASE,
)
# Two or more sound elements joined by spaces/tabs only (no newline -- that's Step 2's job)
RUN_PATTERN = re.compile(
    r'(?:<(?:character_sound|ambient_sound)(?:\s[^>]*)?>[^<]*</(?:character_sound|ambient_sound)>[ \t]+)+'
    r'<(?:character_sound|ambient_sound)(?:\s[^>]*)?>[^<]*</(?:character_sound|ambient_sound)>',
    re.IGNORECASE,
)


def is_bare_verb(text: str) -> bool:
    """Return True if *text* is a single bare verb token (e.g. "sagt", "ruft", "schreit"),
    detected via spaCy part-of-speech tagging rather than a fixed word list.
    """
    cleaned = text.strip().rstrip('.,;:!?)»«"\'').strip()
    if not cleaned or ' ' in cleaned:
        return False  # must be a single bare word, not a phrase
    doc = nlp(cleaned)
    if len(doc) != 1:
        return False
    return doc[0].pos_ in ('VERB', 'AUX')


def reclassify_and_merge_speech_runs(content: str) -> tuple[str, int]:
    """Merge whitespace-separated sound-element runs that contain a bare finite verb
    into a single <character_sound> span. Runs without a detected verb are left untouched.

    Returns (cleaned_content, number_of_runs_merged).
    """
    total_merges = 0

    def _process_stage(m: re.Match) -> str:
        nonlocal total_merges
        open_tag, inner, close_tag = m.group(1), m.group(2), m.group(3)

        def _process_run(run_match: re.Match) -> str:
            nonlocal total_merges
            run_text = run_match.group(0)
            elems = list(SOUND_ELEM.finditer(run_text))
            if len(elems) < 2:
                return run_text

            if not any(is_bare_verb(e.group(3)) for e in elems):
                return run_text  # no verb detected -> leave as separate events

            # Reconstruct, preserving original text and original inter-element spacing exactly
            merged_parts = []
            for i, e in enumerate(elems):
                merged_parts.append(e.group(3))
                if i < len(elems) - 1:
                    merged_parts.append(run_text[e.end():elems[i + 1].start()])
            merged_text = ''.join(merged_parts)

            total_merges += 1
            return f'<character_sound>{merged_text}</character_sound>'

        new_inner = RUN_PATTERN.sub(_process_run, inner)
        return open_tag + new_inner + close_tag

    cleaned = STAGE_BLOCK.sub(_process_stage, content)
    return cleaned, total_merges


def reclassify_speech_verb_files(input_folder: Path, output_folder: Path) -> int:
    """Apply speech-verb run reclassification to all XML files. Returns total runs merged."""
    output_folder.mkdir(parents=True, exist_ok=True)
    xml_files = list(input_folder.glob('*.xml'))
    print(f'Reclassifying speech-verb runs in {len(xml_files)} XML files ...')

    total_merges = 0
    for xml_file in xml_files:
        with open(xml_file, encoding='utf-8') as f:
            content = f.read()

        cleaned, merges = reclassify_and_merge_speech_runs(content)
        total_merges += merges

        out_path = output_folder / xml_file.name
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        if merges:
            print(f'  {xml_file.name}: merged {merges} speech-verb run(s)')

    print(f'\nStep 4 done. {total_merges} speech-verb run(s) reclassified and merged.')
    return total_merges


step4_merged = reclassify_speech_verb_files(PREDICTIONS_FOLDER, OUTPUT_FOLDER)

Loading spaCy German model for finite-verb detection ...
Model loaded.
Reclassifying speech-verb runs in 12 XML files ...

Step 4 done. 0 speech-verb run(s) reclassified and merged.


## 6. Summary

In [202]:
n_after = count_sound_tags(PREDICTIONS_FOLDER)

print('=' * 60)
print('POSTPROCESSING SUMMARY')
print('=' * 60)
print(f'  Files processed                          : {n_files}')
print(f'  Sound annotations before postprocessing  : {n_before}')
print(f'  Step 1 — false positives removed         : {step1_removed}')
print(f'  Step 2 — annotation pairs merged         : {step2_merged}')
print(f'  Step 3 — speaker-wrapped tags removed    : {step3_removed}')
print(f'  Step 4 — speech-verb runs merged         : {step4_merged}')
print(f'  Sound annotations after postprocessing   : {n_after}')
print('=' * 60)
print(f'\nPostprocessed files saved to: {OUTPUT_FOLDER}')
print('You can now run 02_evaluation.ipynb on this folder.')

POSTPROCESSING SUMMARY
  Files processed                          : 12
  Sound annotations before postprocessing  : 1836
  Step 1 — false positives removed         : 166
  Step 2 — annotation pairs merged         : 82
  Step 3 — speaker-wrapped tags removed    : 247
  Step 4 — speech-verb runs merged         : 0
  Sound annotations after postprocessing   : 1341

Postprocessed files saved to: /Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_all_prose_model_predicted_plays_augmented/exp_all_prose_predicted_augmented_postprocessed
You can now run 02_evaluation.ipynb on this folder.
